In [5]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(r"..\Data\sample_data.csv")
df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,cloud_cover,precipitation,us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,month,hour
0,2025-08-01 00:00:00,31.4,74,5.8,982.1,13,0.0,152,77.3,122.9,378.0,24.8,5.4,49.0,86.0,8,0
1,2025-08-01 01:00:00,30.9,77,6.7,981.8,84,0.0,153,75.0,120.1,342.0,22.9,5.1,46.0,90.0,8,1
2,2025-08-01 02:00:00,30.6,77,6.1,981.4,100,0.0,153,73.8,122.0,315.0,21.9,4.9,42.0,93.0,8,2
3,2025-08-01 03:00:00,30.4,76,5.5,981.1,100,0.0,154,74.0,129.5,293.0,22.8,4.8,37.0,92.0,8,3
4,2025-08-01 04:00:00,30.1,74,6.1,981.3,100,0.0,154,70.1,123.4,278.0,24.5,4.8,32.0,90.0,8,4


In [11]:
df = df.sort_values("time").reset_index(drop=True)

df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["day_of_year"] = df["time"].dt.dayofyear
df["day_of_week"] = df["time"].dt.dayofweek 

# Cyclical encoding — so Dec and Jan (both winter) end up numerically close
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print(df[["time", "hour", "month", "month_sin", "month_cos", "hour_sin", "hour_cos"]].head())

                 time  hour  month  month_sin  month_cos  hour_sin  hour_cos
0 2025-08-01 00:00:00     0      8  -0.866025       -0.5  0.000000  1.000000
1 2025-08-01 01:00:00     1      8  -0.866025       -0.5  0.258819  0.965926
2 2025-08-01 02:00:00     2      8  -0.866025       -0.5  0.500000  0.866025
3 2025-08-01 03:00:00     3      8  -0.866025       -0.5  0.707107  0.707107
4 2025-08-01 04:00:00     4      8  -0.866025       -0.5  0.866025  0.500000


In [12]:
lag_hours = [1, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)
    df[f"wind_speed_lag_{lag}h"] = df["wind_speed_10m"].shift(lag)
    df[f"pressure_lag_{lag}h"] = df["surface_pressure"].shift(lag)

print(df[["time", "us_aqi", "aqi_lag_1h", "aqi_lag_6h", "aqi_lag_24h"]].head(30))

                  time  us_aqi  aqi_lag_1h  aqi_lag_6h  aqi_lag_24h
0  2025-08-01 00:00:00     152         NaN         NaN          NaN
1  2025-08-01 01:00:00     153       152.0         NaN          NaN
2  2025-08-01 02:00:00     153       153.0         NaN          NaN
3  2025-08-01 03:00:00     154       153.0         NaN          NaN
4  2025-08-01 04:00:00     154       154.0         NaN          NaN
5  2025-08-01 05:00:00     155       154.0         NaN          NaN
6  2025-08-01 06:00:00     154       155.0       152.0          NaN
7  2025-08-01 07:00:00     154       154.0       153.0          NaN
8  2025-08-01 08:00:00     154       154.0       153.0          NaN
9  2025-08-01 09:00:00     154       154.0       154.0          NaN
10 2025-08-01 10:00:00     154       154.0       154.0          NaN
11 2025-08-01 11:00:00     154       154.0       155.0          NaN
12 2025-08-01 12:00:00     154       154.0       154.0          NaN
13 2025-08-01 13:00:00     153       154.0      

In [13]:
# Rolling averages — smoothed recent trend
df["aqi_roll_mean_6h"] = df["us_aqi"].rolling(window=6).mean()
df["aqi_roll_mean_24h"] = df["us_aqi"].rolling(window=24).mean()
df["aqi_roll_std_24h"] = df["us_aqi"].rolling(window=24).std()

# AQI change rate — is it rising or falling, and how fast
df["aqi_change_1h"] = df["us_aqi"].diff(1)
df["aqi_change_6h"] = df["us_aqi"].diff(6)

print(df[["time", "us_aqi", "aqi_roll_mean_6h", "aqi_roll_mean_24h", "aqi_change_1h"]].head(30))

                  time  us_aqi  aqi_roll_mean_6h  aqi_roll_mean_24h  \
0  2025-08-01 00:00:00     152               NaN                NaN   
1  2025-08-01 01:00:00     153               NaN                NaN   
2  2025-08-01 02:00:00     153               NaN                NaN   
3  2025-08-01 03:00:00     154               NaN                NaN   
4  2025-08-01 04:00:00     154               NaN                NaN   
5  2025-08-01 05:00:00     155        153.500000                NaN   
6  2025-08-01 06:00:00     154        153.833333                NaN   
7  2025-08-01 07:00:00     154        154.000000                NaN   
8  2025-08-01 08:00:00     154        154.166667                NaN   
9  2025-08-01 09:00:00     154        154.166667                NaN   
10 2025-08-01 10:00:00     154        154.166667                NaN   
11 2025-08-01 11:00:00     154        154.000000                NaN   
12 2025-08-01 12:00:00     154        154.000000                NaN   
13 202

In [14]:
print("Shape before cleanup:", df.shape)
print("Rows with any NaN:", df.isnull().any(axis=1).sum())

df_clean = df.dropna().reset_index(drop=True)

print("Shape after cleanup:", df_clean.shape)

Shape before cleanup: (8760, 44)
Rows with any NaN: 24
Shape after cleanup: (8736, 44)


In [15]:
horizon = 72  # hours ahead (3 days)

df_clean["target_aqi_72h"] = df_clean["us_aqi"].shift(-horizon)

print(df_clean[["time", "us_aqi", "target_aqi_72h"]].head(10))
print(df_clean[["time", "us_aqi", "target_aqi_72h"]].tail(10))

                 time  us_aqi  target_aqi_72h
0 2025-08-02 00:00:00     136            94.0
1 2025-08-02 01:00:00     132            92.0
2 2025-08-02 02:00:00     128            91.0
3 2025-08-02 03:00:00     125            90.0
4 2025-08-02 04:00:00     123            90.0
5 2025-08-02 05:00:00     120            89.0
6 2025-08-02 06:00:00     119            89.0
7 2025-08-02 07:00:00     117            89.0
8 2025-08-02 08:00:00     116            89.0
9 2025-08-02 09:00:00     115            89.0
                    time  us_aqi  target_aqi_72h
8726 2026-07-31 14:00:00     155             NaN
8727 2026-07-31 15:00:00     156             NaN
8728 2026-07-31 16:00:00     156             NaN
8729 2026-07-31 17:00:00     157             NaN
8730 2026-07-31 18:00:00     157             NaN
8731 2026-07-31 19:00:00     157             NaN
8732 2026-07-31 20:00:00     157             NaN
8733 2026-07-31 21:00:00     157             NaN
8734 2026-07-31 22:00:00     156             NaN
8735

In [16]:
print("Shape before target cleanup:", df_clean.shape)

df_final = df_clean.dropna(subset=["target_aqi_72h"]).reset_index(drop=True)

print("Shape after target cleanup:", df_final.shape)
print("Expected rows lost:", horizon)

Shape before target cleanup: (8736, 45)
Shape after target cleanup: (8664, 45)
Expected rows lost: 72
